In [1]:
import os
import pandas as pd
from PIL import Image
import hashlib
from tqdm import tqdm

In [2]:
base_path = "/content/drive/MyDrive/capstone/dataset/campur indo/Dataset ocr 1.v2-campuran-indo.tensorflow"
folders = ["train", "valid", "test"]

## Fungsi Cek Gambar Corrupt

In [4]:
def is_valid_image(path):
    try:
        img = Image.open(path)
        img.verify()
        return True
    except:
        return False

## Fungsi Hash (Duplikat)

In [5]:
def get_hash(path):
    try:
        with open(path, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    except:
        return None

## PROSES CLEANING

In [7]:
data = []
hashes = set()

folders = ['train', 'valid', 'test']

corrupt_count = 0
duplicate_count = 0
total_checked = 0

corrupt_files = []
duplicate_files = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)

    for file in tqdm(os.listdir(folder_path)):
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):

            total_checked += 1
            path = os.path.join(folder_path, file)

            # 1. Cek corrupt
            if not is_valid_image(path):
                corrupt_count += 1
                corrupt_files.append(path)
                continue

            # 2. Cek duplikat
            file_hash = get_hash(path)
            if file_hash is None:
                continue

            if file_hash in hashes:
                duplicate_count += 1
                duplicate_files.append(path)
                continue

            hashes.add(file_hash)

            # 3. Simpan data bersih
            data.append({
                "image": file,
                "path": path,
                "label": folder
            })

100%|██████████| 41/41 [00:00<00:00, 144.70it/s]


In [9]:
df = pd.DataFrame(data)

save_path = "/content/drive/MyDrive/capstone/dataset/campur indo/dataset_final.csv"
df.to_csv(save_path, index=False)

In [10]:
print("\n===== HASIL DATA CLEANING =====")
print("Total file dicek     :", total_checked)
print("Gambar corrupt       :", corrupt_count)
print("Data duplikat        :", duplicate_count)
print("Dataset final bersih :", len(df))
print("File CSV disimpan di :", save_path)


===== HASIL DATA CLEANING =====
Total file dicek     : 404
Gambar corrupt       : 0
Data duplikat        : 0
Dataset final bersih : 404
File CSV disimpan di : /content/drive/MyDrive/capstone/dataset/campur indo/dataset_final.csv


In [11]:
print("=== CONTOH CORRUPT ===")
print(corrupt_files[:5])

print("\n=== CONTOH DUPLIKAT ===")
print(duplicate_files[:5])

=== CONTOH CORRUPT ===
[]

=== CONTOH DUPLIKAT ===
[]


In [12]:
print("Jumlah corrupt :", len(corrupt_files))
print("Jumlah duplikat:", len(duplicate_files))

Jumlah corrupt : 0
Jumlah duplikat: 0


In [13]:
pd.DataFrame(corrupt_files, columns=["corrupt_path"]).to_csv("/content/corrupt_files.csv", index=False)
pd.DataFrame(duplicate_files, columns=["duplicate_path"]).to_csv("/content/duplicate_files.csv", index=False)

print("✅ File corrupt & duplikat sudah disimpan")

✅ File corrupt & duplikat sudah disimpan


In [14]:
from PIL import Image
import matplotlib.pyplot as plt

def show_images(file_list, title, max_show=5):
    plt.figure(figsize=(15,5))
    for i, path in enumerate(file_list[:max_show]):
        try:
            img = Image.open(path)
            plt.subplot(1, max_show, i+1)
            plt.imshow(img)
            plt.title(title)
            plt.axis('off')
        except:
            print("Tidak bisa dibuka:", path)
    plt.show()

show_images(corrupt_files, "Corrupt Image")

<Figure size 1500x500 with 0 Axes>

In [15]:
show_images(duplicate_files, "Duplicate Image")

<Figure size 1500x500 with 0 Axes>

In [16]:
hash_map = {}

for item in data:
    h = get_hash(item['path'])
    if h not in hash_map:
        hash_map[h] = []
    hash_map[h].append(item['path'])

dupes = [v for v in hash_map.values() if len(v) > 1]

print("Jumlah grup duplikat:", len(dupes))

for group in dupes[:3]:
    print("\nDuplikat:")
    for g in group:
        print(g)

Jumlah grup duplikat: 0
